# LeetCode 3345 — Smallest Divisible Digit Product I bằng Python

Link đề: https://leetcode.com/problems/smallest-divisible-digit-product-i/ · Độ khó: **Easy** · Chủ đề: Math, Enumeration (liệt kê)

Skeleton LeetCode:

```python
class Solution:
    def smallestNumber(self, n: int, t: int) -> int:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn **toàn bộ** `100 × 10 = 1000` cặp `(n, t)` hợp lệ). Hai ví dụ chính lấy từ đề; ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai số nguyên `n` và `t`; tìm số **nhỏ nhất** `>= n` sao cho **tích các chữ số** của nó **chia hết cho `t`**.
2. Nắm các công cụ Python: tách chữ số bằng `% 10` và `// 10`, vòng lặp `while`, vòng lặp vô hạn `for i in count(n)` / `while True`, `str(i)` + `int(ch)`, `math.prod`, và toán tử chia hết `p % t == 0`.
3. Đi từ cách **đổi sang chuỗi rồi nhân** → **tách chữ số bằng toán** → **`math.prod` trên các chữ số** → **chặn trên `range(n, n + 10)`** → **bảng tích tính sẵn**.
4. Hiểu mấu chốt: trong **10 số liên tiếp** luôn có một số tận cùng bằng `0`; tích chữ số của nó bằng `0`, mà `0` chia hết cho mọi `t`. Vì thế vòng lặp liệt kê **không bao giờ chạy quá 10 bước** — không cần cách xây số thông minh.
5. Tránh các lỗi kinh điển: khởi tạo tích bằng `0` (thay vì `1`), bỏ qua chữ số `0` khi nhân, bắt đầu duyệt từ `n + 1` (bỏ sót chính `n`), và quên rằng `0 % t == 0`.

## 2. Đề bài & ví dụ

Cho hai số nguyên `n` và `t`. Trả về số **nhỏ nhất** lớn hơn hoặc bằng `n` sao cho **tích các chữ số** của nó chia hết cho `t`.

**Ví dụ 1**

- Input: `n = 10, t = 2`
- Output: `10`
- Giải thích: các chữ số của `10` là `1` và `0`, tích `1 × 0 = 0`. Mà `0` chia hết cho `2`, nên `10` thoả mãn và nó cũng là số nhỏ nhất `>= 10`.

**Ví dụ 2**

- Input: `n = 15, t = 3`
- Output: `16`
- Giải thích: `15` có tích `1 × 5 = 5`, không chia hết cho `3`. `16` có tích `1 × 6 = 6`, chia hết cho `3`.

**Ràng buộc**

- `1 <= n <= 100`
- `1 <= t <= 10`

Ràng buộc nhỏ như vậy gợi ý ngay: **liệt kê thử từng số** là đủ.

In [1]:
import random
from math import prod

examples = [
    (10, 2, 10),
    (15, 3, 16),
]

def digit_product_ref(x: int) -> int:
    """Tích các chữ số của x (bản tham chiếu dùng chuỗi)."""
    return prod(int(ch) for ch in str(x))

for n, t, expected in examples:
    x = n
    while digit_product_ref(x) % t != 0:
        x += 1
    print(f"n={n}, t={t} -> {x} (kỳ vọng {expected})")
    assert x == expected

n=10, t=2 -> 10 (kỳ vọng 10)
n=15, t=3 -> 16 (kỳ vọng 16)


## 3. Kiến thức nền

### 3.1. Tách chữ số bằng `% 10` và `// 10`

- `x % 10` là **chữ số cuối** của `x` (phần dư khi chia cho `10`).
- `x // 10` là `x` **bỏ chữ số cuối** (chia lấy phần nguyên).

Lặp hai thao tác đó cho tới khi `x` về `0`, ta lấy ra từng chữ số **từ phải sang trái**.

In [2]:
x = 1234
print(x % 10)    # 4  : chữ số cuối
print(x // 10)   # 123: bỏ chữ số cuối
print(x / 10)    # 123.4 (phép chia thực, KHÔNG dùng để tách chữ số)

x = 1234
while x:                     # x khác 0 thì còn chữ số
    print("chữ số:", x % 10)
    x //= 10

4
123
123.4
chữ số: 4
chữ số: 3
chữ số: 2
chữ số: 1


### 3.2. Tích các chữ số và vì sao bắt đầu từ `p = 1`

Tích là **phép nhân dồn**: `p = p * chữ_số`. Phần tử trung lập của phép nhân là `1` (nhân với `1` không đổi), nên ta khởi tạo `p = 1` — **không phải `0`** (nếu bắt đầu bằng `0` thì mọi tích đều bằng `0`).

Ví dụ `234 -> 2 × 3 × 4 = 24`. Nếu có chữ số `0` thì tích **bằng `0`**: `105 -> 1 × 0 × 5 = 0`.

In [3]:
def digit_product(x: int) -> int:
    p = 1
    while x:
        p *= x % 10
        x //= 10
    return p

print(digit_product(234))   # 24
print(digit_product(105))   # 0  (có chữ số 0)
print(digit_product(7))     # 7
print(digit_product(100))   # 0

24
0
7
0


### 3.3. Chia hết và số `0`

`p % t == 0` nghĩa là `p` chia hết cho `t`. Với `p = 0`: `0 % t == 0` với **mọi** `t >= 1`, vì `0 = t × 0`. Do đó **tích bằng `0` luôn thoả mãn đề** — đây là chìa khoá của bài.

In [4]:
for t in (1, 2, 3, 7, 10):
    print(f"0 % {t} = {0 % t}   ->  0 chia hết cho {t}? {0 % t == 0}")

print(24 % 3 == 0, 24 % 5 == 0)   # True False
print(6 % 3 == 0, 5 % 3 == 0)     # True False

0 % 1 = 0   ->  0 chia hết cho 1? True
0 % 2 = 0   ->  0 chia hết cho 2? True
0 % 3 = 0   ->  0 chia hết cho 3? True
0 % 7 = 0   ->  0 chia hết cho 7? True
0 % 10 = 0   ->  0 chia hết cho 10? True
True False
True False


### 3.4. Vòng lặp "chưa biết dừng ở đâu": `while True` và `itertools.count`

Ta không biết trước mình phải thử bao nhiêu số, nhưng biết **dừng ngay khi gặp số thoả mãn**. Có hai cách viết:

- `while True:` ... `return` khi thấy kết quả, còn không thì `i += 1`.
- `for i in count(n):` — `count(n)` sinh `n, n+1, n+2, ...` **vô hạn**, và ta `return` bên trong để dừng.

Cả hai tương đương; LeetCode đảm bảo luôn có đáp án, nên vòng lặp luôn kết thúc.

In [5]:
from itertools import count

for i in count(5):
    print("thử", i)
    if i >= 8:       # điều kiện dừng giả lập
        break

i = 5
while True:
    print("thử", i)
    if i >= 8:
        break
    i += 1

thử 5
thử 6
thử 7
thử 8
thử 5
thử 6
thử 7
thử 8


### 3.5. Chuỗi: `str(i)`, `int(ch)` và `math.prod`

Cách khác để lấy chữ số: `str(1234)` thành `"1234"`, rồi mỗi ký tự `ch` đổi lại bằng `int(ch)`. `math.prod(iterable)` nhân tất cả phần tử (tích rỗng bằng `1`).

In [6]:
from math import prod

s = str(1234)
print(s, list(s))                       # '1234' ['1', '2', '3', '4']
digits = [int(ch) for ch in s]
print(digits)                           # [1, 2, 3, 4]
print(prod(digits))                     # 24
print(prod([]))                         # 1 (tích rỗng)
print(prod(int(ch) for ch in "105"))    # 0

1234 ['1', '2', '3', '4']
[1, 2, 3, 4]
24
1
0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `n = 15, t = 3`, ta **thử từng số** bắt đầu từ chính `n`:

| số | các chữ số | tích | tích % 3 | thoả? |
|---|---|---|---|---|
| 15 | 1, 5 | 5 | 2 | không |
| 16 | 1, 6 | 6 | 0 | **có** |

Dừng ở `16`. Một ví dụ khác `n = 19, t = 7`:

| số | tích | tích % 7 | thoả? |
|---|---|---|---|
| 19 | 9 | 2 | không |
| 20 | 0 | 0 | **có** |

Để ý: số `20` có chữ số `0` nên tích `0`, chia hết cho mọi `t`. Điều này luôn "cứu" ta sau nhiều nhất vài bước.

In [7]:
n, t = 15, 3
for x in range(n, n + 6):
    p = digit_product(x)
    print(f"{x}: tích={p}, tích % {t} = {p % t}, thoả? {p % t == 0}")

15: tích=5, tích % 3 = 2, thoả? False
16: tích=6, tích % 3 = 0, thoả? True
17: tích=7, tích % 3 = 1, thoả? False
18: tích=8, tích % 3 = 2, thoả? False
19: tích=9, tích % 3 = 0, thoả? True
20: tích=0, tích % 3 = 0, thoả? True


## 5. Cách 1 — Brute Force / baseline: đổi sang chuỗi rồi nhân

**Ý tưởng.** Với `x = n, n+1, n+2, ...`: đổi `x` thành chuỗi, nhân các chữ số, nếu tích chia hết cho `t` thì trả về `x`.

Đây đúng là cách "người mới" nghĩ ra trước: **liệt kê** và **kiểm tra**. Code dùng vòng `for ch in str(x)` nên dễ đọc, chưa cần `% 10`.

In [8]:
def smallest_number_str(n: int, t: int) -> int:
    x = n
    while True:
        p = 1
        for ch in str(x):          # duyệt từng chữ số dạng ký tự
            p *= int(ch)
        if p % t == 0:
            return x
        x += 1

print(smallest_number_str(10, 2))   # 10
print(smallest_number_str(15, 3))   # 16
print(smallest_number_str(100, 7))  # 100
print(smallest_number_str(19, 7))   # 20

10
16
100
20


**Độ phức tạp (Cách 1).** Mỗi số có `O(log n)` chữ số (số chữ số của `x`). Số ứng viên tối đa là `10` (sẽ chứng minh ở mục 6). Vậy tổng `O(log n)` thời gian, `O(1)` bộ nhớ phụ (chuỗi `str(x)` có độ dài `O(log n)`, rất nhỏ).

**Nút thắt.** Thực ra không có nút thắt nào đáng kể! Ràng buộc quá nhỏ nên baseline đã tối ưu về bậc. Phần "cải tiến" ở các cách sau chỉ là gọn hơn, tránh chuỗi, hoặc **chặn trên số bước** để thấy rõ vòng lặp luôn dừng.

## 6. Ý tưởng thuật toán / bước đột phá

**Trong mỗi 10 số nguyên liên tiếp luôn có đúng một số tận cùng bằng `0`.** Số đó có chữ số `0` nên tích chữ số bằng `0`, mà `0 % t == 0`. Vậy trong dãy `n, n+1, ..., n+9` **luôn có ít nhất một số thoả mãn**, nghĩa là đáp án `<= n + 9`.

Hệ quả: vòng lặp liệt kê chạy **tối đa 10 lần**, độ phức tạp là hằng số nhân với số chữ số. Không cần suy luận gì thêm về `t`.

Cell dưới đây kiểm chứng "chặn trên" bằng thực nghiệm: với mọi `(n, t)` hợp lệ, `đáp_án - n` lớn nhất là bao nhiêu?

In [9]:
def gap(n, t):
    return smallest_number_str(n, t) - n

worst = max(((gap(n, t), n, t) for n in range(1, 101) for t in range(1, 11)))
print("bước nhảy lớn nhất:", worst)            # (độ lệch, n, t)
assert worst[0] <= 9

# Với n tận cùng 0 thì đáp án chính là n (tích = 0)
print([smallest_number_str(n, 7) for n in (10, 20, 30, 100)])   # [10, 20, 30, 100]

bước nhảy lớn nhất: (9, 91, 10)
[10, 20, 30, 100]


In [10]:
# Mỗi 10 số liên tiếp có đúng một số chia hết cho 10 (tận cùng 0)
for n in (1, 11, 26, 95):
    window = list(range(n, n + 10))
    ends_with_zero = [x for x in window if x % 10 == 0]
    print(n, "->", ends_with_zero)
    assert len(ends_with_zero) == 1

1 -> [10]
11 -> [20]
26 -> [30]
95 -> [100]


## 7. Các cách tiếp theo

### Cách 2 — Tách chữ số bằng `% 10` và `// 10` (bản chuẩn)

**Giải thích từng đoạn.**

1. `for i in count(n):` — thử `i = n, n+1, ...` vô hạn.
2. `p = 1; x = i` — `p` là tích, `x` là bản sao của `i` để ta cắt dần chữ số mà không làm mất `i`.
3. `while x: p *= x % 10; x //= 10` — nhân chữ số cuối rồi bỏ nó đi.
4. `if p % t == 0: return i` — đủ điều kiện thì trả về, vì ta duyệt tăng dần nên `i` là số nhỏ nhất.

In [11]:
from itertools import count

def smallest_number_digits(n: int, t: int) -> int:
    for i in count(n):
        p = 1
        x = i
        while x:
            p *= x % 10
            x //= 10
        if p % t == 0:
            return i

print(smallest_number_digits(10, 2))   # 10
print(smallest_number_digits(15, 3))   # 16
print(smallest_number_digits(99, 9))   # 100

10
16
99


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from itertools import count

class SolutionDigits:
    def smallestNumber(self, n: int, t: int) -> int:
        for i in count(n):
            p = 1
            x = i
            while x:
                p *= x % 10
                x //= 10
            if p % t == 0:
                return i

print(SolutionDigits().smallestNumber(15, 3))   # 16

16


**Độ phức tạp Cách 2:** **Time `O(log n)`** (tối đa 10 ứng viên, mỗi ứng viên `O(log n)` chữ số), **Space `O(1)`**. Đây là lời giải chính thức của đề.

### Cách 3 — `math.prod` trên các chữ số

**Giải thích.** Thay vòng `while` tách chữ số bằng một dòng: `prod(map(int, str(i)))`. `map(int, "16")` cho `1, 6`; `prod` nhân chúng lại. Dùng `next(...)` kèm generator để tìm số đầu tiên thoả điều kiện — gọn nhất nhưng hơi "cao tay", hợp khi bạn đã quen generator.

In [13]:
from itertools import count
from math import prod

def smallest_number_prod(n: int, t: int) -> int:
    return next(i for i in count(n) if prod(map(int, str(i))) % t == 0)

print(smallest_number_prod(10, 2))   # 10
print(smallest_number_prod(15, 3))   # 16
print(smallest_number_prod(19, 7))   # 20

10
16
20


In [14]:
# Code nộp LeetCode cho Cách 3
from itertools import count
from math import prod

class SolutionProd:
    def smallestNumber(self, n: int, t: int) -> int:
        return next(i for i in count(n) if prod(map(int, str(i))) % t == 0)

print(SolutionProd().smallestNumber(19, 7))   # 20

20


**Độ phức tạp Cách 3:** **Time `O(log n)`**, **Space `O(log n)`** cho chuỗi `str(i)` tạm (thực tế rất nhỏ).

### Cách 4 — Chặn trên: `for i in range(n, n + 10)`

**Giải thích.** Nhờ bước đột phá, ta thay vòng "vô hạn" bằng `range(n, n + 10)` — **chắc chắn** có đáp án trong khoảng này. Cách này cho thấy bài **không thể lặp vô hạn**. Ta cũng dừng sớm khi gặp chữ số `0` (tích `0`) để khỏi nhân thừa.

In [15]:
def smallest_number_bounded(n: int, t: int) -> int:
    for i in range(n, n + 10):          # luôn có đáp án trong 10 số này
        p = 1
        x = i
        while x:
            d = x % 10
            if d == 0:                   # có chữ số 0 -> tích = 0, chia hết cho mọi t
                p = 0
                break
            p *= d
            x //= 10
        if p % t == 0:
            return i
    raise AssertionError("không thể xảy ra: trong 10 số liên tiếp luôn có đáp án")

print(smallest_number_bounded(10, 2))   # 10
print(smallest_number_bounded(15, 3))   # 16
print(smallest_number_bounded(19, 7))   # 20

10
16
20


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionBounded:
    def smallestNumber(self, n: int, t: int) -> int:
        for i in range(n, n + 10):
            p = 1
            x = i
            while x:
                d = x % 10
                if d == 0:
                    p = 0
                    break
                p *= d
                x //= 10
            if p % t == 0:
                return i
        return -1   # không bao giờ tới đây

print(SolutionBounded().smallestNumber(19, 7))   # 20

20


**Độ phức tạp Cách 4:** **Time `O(log n)`** (đúng `<= 10` lần lặp ngoài), **Space `O(1)`**.

### Cách 5 — Bảng tích tính sẵn (tra bảng)

**Giải thích.** Vì `n <= 100`, đáp án `<= 109`. Ta tính trước `table[x] = tích chữ số của x` cho `x = 0..109` bằng một công thức đệ quy gọn: `table[x] = table[x // 10] * (x % 10)` (với `x < 10` thì `table[x] = x`). Sau đó chỉ việc tìm `x` nhỏ nhất `>= n` có `table[x] % t == 0`. Cách này hợp khi phải trả lời **nhiều truy vấn** trên cùng miền nhỏ.

In [17]:
LIMIT = 120
table = [0] * LIMIT
for x in range(1, LIMIT):
    table[x] = x if x < 10 else table[x // 10] * (x % 10)

print(table[:25])   # [0, 1, 2, ..., 9, 0, 1, 2, ...]

def smallest_number_table(n: int, t: int) -> int:
    x = n
    while table[x] % t != 0:
        x += 1
    return x

print(smallest_number_table(10, 2))   # 10
print(smallest_number_table(15, 3))   # 16
print(smallest_number_table(19, 7))   # 20

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 2, 4, 6, 8]
10
16
20


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionTable:
    def smallestNumber(self, n: int, t: int) -> int:
        table = [0] * 130
        for x in range(1, 130):
            table[x] = x if x < 10 else table[x // 10] * (x % 10)
        while table[n] % t != 0:
            n += 1
        return n

print(SolutionTable().smallestNumber(15, 3))   # 16

16


**Độ phức tạp Cách 5:** **Time `O(M)`** để dựng bảng (`M = 130` là hằng số) rồi `O(1)` mỗi truy vấn (tối đa 10 bước); **Space `O(M)`**. Với một truy vấn đơn lẻ thì không lợi hơn cách 2, nhưng nó minh hoạ kỹ thuật "tính trước".

## 8. TRACE từng bước

Ta trace Cách 2 trên `n = 13, t = 7` (cần chữ số `7` hoặc `0`...) và `n = 15, t = 3`. Mỗi vòng ngoài in: `i`, từng chữ số lấy ra, tích `p` sau khi nhân, rồi quyết định.

In [19]:
def trace_smallest_number(n: int, t: int) -> int:
    print(f"n = {n}, t = {t}")
    for i in count(n):
        p = 1
        x = i
        steps = []
        while x:
            d = x % 10
            p *= d
            steps.append(d)
            x //= 10
        ok = (p % t == 0)
        print(f"  i={i:>3}, chữ số (phải->trái)={steps}, tích p={p:>3}, p % t={p % t}, thoả? {ok}")
        if ok:
            print("  => trả về", i)
            return i

trace_smallest_number(15, 3)
print()
trace_smallest_number(13, 7)
print()
trace_smallest_number(99, 2)

n = 15, t = 3
  i= 15, chữ số (phải->trái)=[5, 1], tích p=  5, p % t=2, thoả? False
  i= 16, chữ số (phải->trái)=[6, 1], tích p=  6, p % t=0, thoả? True
  => trả về 16

n = 13, t = 7
  i= 13, chữ số (phải->trái)=[3, 1], tích p=  3, p % t=3, thoả? False
  i= 14, chữ số (phải->trái)=[4, 1], tích p=  4, p % t=4, thoả? False
  i= 15, chữ số (phải->trái)=[5, 1], tích p=  5, p % t=5, thoả? False
  i= 16, chữ số (phải->trái)=[6, 1], tích p=  6, p % t=6, thoả? False
  i= 17, chữ số (phải->trái)=[7, 1], tích p=  7, p % t=0, thoả? True
  => trả về 17

n = 99, t = 2
  i= 99, chữ số (phải->trái)=[9, 9], tích p= 81, p % t=1, thoả? False
  i=100, chữ số (phải->trái)=[0, 0, 1], tích p=  0, p % t=0, thoả? True
  => trả về 100


100

Bảng trace cho `n = 13, t = 7`:

| i | các chữ số | tích `p` | `p % 7` | thoả? |
|---|---|---|---|---|
| 13 | 1, 3 | 3 | 3 | không |
| 14 | 1, 4 | 4 | 4 | không |
| 15 | 1, 5 | 5 | 5 | không |
| 16 | 1, 6 | 6 | 6 | không |
| 17 | 1, 7 | 7 | 0 | **có** → trả về `17` |

Còn `n = 99, t = 2`: `99` có tích `81` (lẻ) nên không chia hết cho `2`, `100` có tích `0` nên thoả → `100`. Trace in ra đúng như vậy ở trên. Chú ý: số mới `100` thêm một chữ số — vòng `while x` vẫn xử lý bình thường.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (chuẩn, dễ đọc). Trên LeetCode `count` đã nhập sẵn trong môi trường; khi viết ngoài, nhớ `from itertools import count`.

In [20]:
from itertools import count

class Solution:
    def smallestNumber(self, n: int, t: int) -> int:
        for i in count(n):
            p = 1
            x = i
            while x:
                p *= x % 10
                x //= 10
            if p % t == 0:
                return i

sol = Solution()
print(sol.smallestNumber(10, 2))   # 10
print(sol.smallestNumber(15, 3))   # 16

10
16


## 10. Vì sao thuật toán đúng

- **Duyệt tăng dần từ `n`:** ta thử lần lượt `n, n+1, ...` nên số đầu tiên thoả mãn **chính là số nhỏ nhất** `>= n` thoả mãn (không bỏ sót số nhỏ hơn).
- **Luôn kết thúc:** trong `n..n+9` có một số tận cùng `0`, tích chữ số bằng `0`, chia hết cho `t`. Do đó vòng lặp dừng sau tối đa `10` lần thử.
- **Tích đúng:** `x % 10` là chữ số cuối, `x // 10` bỏ chữ số cuối; lặp tới `x = 0` thì đã nhân hết mọi chữ số đúng một lần. Khởi tạo `p = 1` để phép nhân không bị triệt tiêu từ đầu.
- **Chữ số `0` được xử lý đúng:** `x % 10 = 0` nhân vào làm `p = 0` và `p` giữ nguyên `0` (không "hồi phục"), đúng với định nghĩa tích.

## 11. Phân tích Time / Space Complexity

Gọi `d` là số chữ số của đáp án (`d <= 3` ở đây, nói chung `d = O(log n)`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1 — chuỗi + nhân | `O(10 · d)` = `O(log n)` | `O(d)` | dễ đọc nhất |
| 2 — `% 10`, `// 10` | `O(10 · d)` = `O(log n)` | `O(1)` | lời giải chính thức |
| 3 — `prod(map(int, str(i)))` | `O(10 · d)` | `O(d)` | ngắn gọn |
| 4 — `range(n, n + 10)` | `O(10 · d)` | `O(1)` | chặn trên rõ ràng |
| 5 — bảng tính sẵn | dựng `O(M)`, mỗi truy vấn `O(10)` | `O(M)` | hợp khi nhiều truy vấn |

Vì `n <= 100` và `t <= 10` nên mọi cách đều chạy tức thì; con số `10` ứng viên là **hằng số**, nên Big-O thường viết là `O(log n)` (theo số chữ số) hoặc `O(1)` với ràng buộc này.

## 12. Bộ test

Gom các cách vào `impls` và so với kỳ vọng.

In [21]:
class SolutionPlain:
    """Bản tham chiếu: vét cạn dùng chuỗi (Cách 1)."""
    def smallestNumber(self, n: int, t: int) -> int:
        return smallest_number_str(n, t)

impls = {
    "str": smallest_number_str,
    "digits": smallest_number_digits,
    "prod": smallest_number_prod,
    "bounded": smallest_number_bounded,
    "table": smallest_number_table,
    "Solution": Solution().smallestNumber,
    "SolutionDigits": SolutionDigits().smallestNumber,
    "SolutionProd": SolutionProd().smallestNumber,
    "SolutionBounded": SolutionBounded().smallestNumber,
    "SolutionTable": SolutionTable().smallestNumber,
}

def check_all(n, t, expected, note=""):
    for name, f in impls.items():
        got = f(n, t)
        assert got == expected, (name, n, t, got, expected)
    print(f"OK  n={n}, t={t} -> {expected}  {note}")

check_all(10, 2, 10, "ví dụ 1")
check_all(15, 3, 16, "ví dụ 2")

OK  n=10, t=2 -> 10  ví dụ 1
OK  n=15, t=3 -> 16  ví dụ 2


### 12.1. Các trường hợp biên

In [22]:
check_all(1, 1, 1, "t = 1 thì tích nào cũng chia hết")
check_all(1, 10, 10, "n nhỏ nhất, t lớn nhất: phải tới 10 (tích 0)")
check_all(100, 10, 100, "n = 100 (tích 0) luôn thoả")
check_all(100, 1, 100, "n lớn nhất, t = 1")
check_all(11, 10, 20, "tích 1..9 đều không chia hết cho 10 (11..19) -> 20")
check_all(19, 7, 20, "tích 9 không chia hết cho 7, 20 thì có")
check_all(9, 2, 10, "9 lẻ -> 10 (tích 0)")
check_all(8, 4, 8, "một chữ số: 8 chia hết cho 4")
check_all(99, 2, 100, "99 -> tích 81 lẻ, 100 tích 0")
check_all(25, 5, 25, "tích 10 chia hết cho 5")
check_all(23, 4, 24, "23: tích 6 không chia hết 4, 24: tích 8 chia hết 4 (tự dựng)")

OK  n=1, t=1 -> 1  t = 1 thì tích nào cũng chia hết
OK  n=1, t=10 -> 10  n nhỏ nhất, t lớn nhất: phải tới 10 (tích 0)
OK  n=100, t=10 -> 100  n = 100 (tích 0) luôn thoả
OK  n=100, t=1 -> 100  n lớn nhất, t = 1
OK  n=11, t=10 -> 20  tích 1..9 đều không chia hết cho 10 (11..19) -> 20
OK  n=19, t=7 -> 20  tích 9 không chia hết cho 7, 20 thì có
OK  n=9, t=2 -> 10  9 lẻ -> 10 (tích 0)
OK  n=8, t=4 -> 8  một chữ số: 8 chia hết cho 4
OK  n=99, t=2 -> 100  99 -> tích 81 lẻ, 100 tích 0
OK  n=25, t=5 -> 25  tích 10 chia hết cho 5
OK  n=23, t=4 -> 24  23: tích 6 không chia hết 4, 24: tích 8 chia hết 4 (tự dựng)


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách "gần đúng" hay gặp: (a) **bỏ qua chữ số `0`** (coi `0` như không có) và (b) **bắt đầu từ `n + 1`**. Ta tìm input làm chúng lệch.

In [23]:
def wrong_skip_zero(n, t):
    x = n
    while True:
        p = 1
        for ch in str(x):
            if ch != "0":          # SAI: bỏ chữ số 0
                p *= int(ch)
        if p % t == 0:
            return x
        x += 1

def wrong_start_next(n, t):
    x = n + 1                       # SAI: bỏ sót chính n
    while True:
        p = 1
        for ch in str(x):
            p *= int(ch)
        if p % t == 0:
            return x
        x += 1

bad1 = [(n, t) for n in range(1, 101) for t in range(1, 11) if wrong_skip_zero(n, t) != smallest_number_str(n, t)]
bad2 = [(n, t) for n in range(1, 101) for t in range(1, 11) if wrong_start_next(n, t) != smallest_number_str(n, t)]
print("bỏ qua chữ số 0 sai ở", len(bad1), "cặp, ví dụ:", bad1[:4])
print("bắt đầu từ n+1 sai ở", len(bad2), "cặp, ví dụ:", bad2[:4])
assert bad1 and bad2
print(wrong_skip_zero(10, 2), "(sai) vs", smallest_number_str(10, 2), "(đúng)")
print(wrong_start_next(10, 2), "(sai) vs", smallest_number_str(10, 2), "(đúng)")

bỏ qua chữ số 0 sai ở 213 cặp, ví dụ: [(1, 10), (2, 10), (3, 10), (4, 10)]
bắt đầu từ n+1 sai ở 455 cặp, ví dụ: [(1, 1), (2, 1), (2, 2), (3, 1)]
12 (sai) vs 10 (đúng)
12 (sai) vs 10 (đúng)


### 12.3. Đối chiếu toàn bộ và ngẫu nhiên (có seed)

Miền input chỉ có `100 × 10 = 1000` cặp, nên ta **đối chiếu tất cả** các cách với nhau. Thêm một lượt ngẫu nhiên có `seed` trong đúng ràng buộc để minh hoạ cách làm test ngẫu nhiên.

In [24]:
# Đối chiếu toàn bộ 1000 cặp (n, t)
for n in range(1, 101):
    for t in range(1, 11):
        results = {name: f(n, t) for name, f in impls.items()}
        assert len(set(results.values())) == 1, (n, t, results)
        ans = results["str"]
        # kiểm tra tính đúng độc lập: ans thoả, và mọi số trong [n, ans) không thoả
        assert digit_product_ref(ans) % t == 0
        for x in range(n, ans):
            assert digit_product_ref(x) % t != 0
        assert ans - n <= 9
print("Đối chiếu đủ 1000 cặp: OK")

rng = random.Random(3345)
for _ in range(500):
    n = rng.randint(1, 100)
    t = rng.randint(1, 10)
    vals = {f(n, t) for f in impls.values()}
    assert len(vals) == 1
print("Ngẫu nhiên có seed: OK")

Đối chiếu đủ 1000 cặp: OK
Ngẫu nhiên có seed: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Khởi tạo tích `p = 0`

Nếu bắt đầu `p = 0` thì mọi tích đều bằng `0`, và hàm luôn trả về chính `n` (vì `0 % t == 0`).

In [25]:
def sai_p_bang_0(n, t):
    for i in count(n):
        p = 0                  # SAI: phải là 1
        x = i
        while x:
            p *= x % 10
            x //= 10
        if p % t == 0:
            return i

print(sai_p_bang_0(15, 3), "(sai, luôn trả n) — kỳ vọng 16")
assert sai_p_bang_0(15, 3) == 15 != 16

15 (sai, luôn trả n) — kỳ vọng 16


**Vì sao sai:** `0` là phần tử **triệt tiêu** của phép nhân, không phải phần tử trung lập. Tích phải bắt đầu từ `1`. **Sửa:** `p = 1` (xem mục 3.2).

### Lỗi 2: Dùng `/` thay `//` để bỏ chữ số cuối

In [26]:
def sai_chia_thuc(n, t, max_steps=6):
    # minh hoạ: x /= 10 cho số thực nên while x không bao giờ về 0 đúng lúc
    x = float(n)
    out = []
    for _ in range(max_steps):
        out.append(x)
        x /= 10
    return out

print(sai_chia_thuc(15, 3))   # 15.0, 1.5, 0.15, ... không bao giờ về 0 sớm

def dung_chia_nguyen(x):
    out = []
    while x:
        out.append(x)
        x //= 10
    return out

print(dung_chia_nguyen(15))   # [15, 1] rồi dừng

[15.0, 1.5, 0.15, 0.015, 0.0015, 0.00015000000000000001]
[15, 1]


**Vì sao sai:** `/` trả số thực (`15 / 10 = 1.5`), nên `while x` gần như không dừng đúng và `x % 10` cho phần thập phân. **Sửa:** dùng `//` (chia lấy phần nguyên).

### Lỗi 3: Bắt đầu duyệt từ `n + 1`

In [27]:
def sai_bat_dau_n_cong_1(n, t):
    for i in count(n + 1):       # SAI: bỏ sót n
        p = 1
        x = i
        while x:
            p *= x % 10
            x //= 10
        if p % t == 0:
            return i

print(sai_bat_dau_n_cong_1(10, 2), "(sai) — đề cho phép đáp án bằng chính n, kỳ vọng 10")
assert sai_bat_dau_n_cong_1(10, 2) == 12 != 10

12 (sai) — đề cho phép đáp án bằng chính n, kỳ vọng 10


**Vì sao sai:** đề nói "lớn hơn **hoặc bằng** `n`" (`>= n`). Ví dụ 1 có đáp án `10 = n`. **Sửa:** `count(n)` / `range(n, ...)` bắt đầu từ chính `n`.

### Lỗi 4: Bỏ qua chữ số `0` khi nhân

In [28]:
def sai_bo_so_0(n, t):
    for i in count(n):
        p = 1
        x = i
        while x:
            d = x % 10
            if d != 0:           # SAI: bỏ chữ số 0
                p *= d
            x //= 10
        if p % t == 0:
            return i

print(sai_bo_so_0(10, 2), "(sai: 10 -> tích 1)  — kỳ vọng 10")
print(sai_bo_so_0(10, 3), "(sai) — kỳ vọng 10")
assert sai_bo_so_0(10, 2) != 10 and smallest_number_digits(10, 2) == 10

12 (sai: 10 -> tích 1)  — kỳ vọng 10
13 (sai) — kỳ vọng 10


**Vì sao sai:** chữ số `0` là một chữ số thật; tích chứa `0` thì bằng `0`. Bỏ nó đi làm tích `10` thành `1`, không chia hết cho `2`. **Sửa:** nhân **mọi** chữ số, kể cả `0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Số nhỏ nhất `>= n` thoả điều kiện" + miền nhỏ → liệt kê từ `n` lên.** Đừng cố xây số bằng toán nếu giới hạn chỉ vài trăm.
- **Tích chữ số + chữ số `0` → tích `0`.** Mỗi khi đề đụng tới "tích các chữ số", hãy nghĩ ngay tới `0` làm tích triệt tiêu.
- **"Trong mọi `k` số liên tiếp có một số chia hết cho `k`"** là mẹo chặn trên rất hay: dùng để chứng minh vòng lặp dừng và ước lượng số bước.
- **Tách chữ số:** `while x: d = x % 10; x //= 10`. Mẫu này xuất hiện ở "Add Digits", "Palindrome Number", "Happy Number"...
- **Khi `n` lên tới `10^16` (bài II)** thì không thể liệt kê: phải phân tích `t` thành thừa số nguyên tố `2, 3, 5, 7` và xây số tham lam. Ở bài I này (`n <= 100`) đừng làm phức tạp.

## 15. Biến thể của bài

1. **Smallest Divisible Digit Product II** (LeetCode 3348): `n` rất lớn (chuỗi tới `10^16`), phải xây số bằng phân tích thừa số nguyên tố — vét cạn không còn đủ.
2. **Tổng chữ số** thay vì tích: "số nhỏ nhất `>= n` có tổng chữ số chia hết cho `t`".
3. **Số lớn nhất `<= n`** có tích chữ số chia hết cho `t` (duyệt ngược).
4. **Đếm** các số trong `[a, b]` có tích chữ số chia hết cho `t`.

In [29]:
def smallest_digit_sum_divisible(n: int, t: int) -> int:
    """Biến thể: số nhỏ nhất >= n có TỔNG chữ số chia hết cho t."""
    for i in count(n):
        s = 0
        x = i
        while x:
            s += x % 10
            x //= 10
        if s % t == 0:
            return i

print(smallest_digit_sum_divisible(15, 3))   # 15 (1+5=6, chia hết cho 3)
print(smallest_digit_sum_divisible(16, 3))   # 18 (1+8=9)
print(smallest_digit_sum_divisible(10, 7))   # 16 (1+6=7)

def largest_at_most(n: int, t: int):
    """Biến thể: số lớn nhất <= n có tích chữ số chia hết cho t (n >= 1)."""
    for i in range(n, 0, -1):
        if digit_product(i) % t == 0:
            return i
    return None

print(largest_at_most(15, 3))   # 13 (15: tích 5, 14: tích 4 đều không; 13: tích 3 chia hết cho 3)

15
18
16
13


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Đổi `str`, nhân từng ký tự | `O(log n)` | `O(log n)` | mới học, dễ đọc |
| 2 | `% 10` + `// 10` | `O(log n)` | `O(1)` | lời giải chuẩn, phỏng vấn |
| 3 | `prod(map(int, str(i)))` + `next` | `O(log n)` | `O(log n)` | viết nhanh, ngắn |
| 4 | `range(n, n + 10)` + dừng sớm ở `0` | `O(log n)` | `O(1)` | muốn chứng minh vòng lặp hữu hạn |
| 5 | Bảng tích tính sẵn | `O(M)` dựng, `O(1)` truy vấn | `O(M)` | nhiều truy vấn, miền nhỏ |

Với ràng buộc của bài, **cách 2** là cân bằng nhất; cách 4 thể hiện rõ nhất "vì sao thuật toán dừng".

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `digit_product(234)`, `digit_product(105)`, `45 % 9`, `smallest_number_digits(99, 9)`, `smallest_number_digits(20, 9)`.

**Bài 2 (tính tay).** Với `n = 23, t = 4`: lập bảng `số — tích — thoả?` cho tới khi tìm được đáp án.

**Bài 3 (điền TODO).** Hoàn thành `smallest_number_todo(n, t)` theo Cách 2.

In [30]:
def smallest_number_todo(n, t):
    # TODO 1: duyệt i = n, n+1, ... (vô hạn)
    # TODO 2: p = 1, x = i
    # TODO 3: while x: nhân chữ số cuối vào p, bỏ chữ số cuối khỏi x
    # TODO 4: nếu p % t == 0 thì trả về i
    pass

# Tự làm xong rồi mới mở test:
# assert smallest_number_todo(10, 2) == 10
# assert smallest_number_todo(15, 3) == 16

**Bài 4 (tự viết lại).** Viết `smallest_number_v2(n, t)` **không dùng `itertools.count`** và **không dùng chuỗi**: chỉ `while True`, `% 10`, `// 10`. Rồi so sánh với `smallest_number_digits` trên mọi `n` từ `1` đến `100`, mọi `t` từ `1` đến `10`.

**Bài 5 (biến thể / mini challenge).** Viết `count_good(a, b, t)` đếm số `x` trong `[a, b]` có tích chữ số chia hết cho `t`; sau đó viết `smallest_digit_sum_divisible(n, t)` (tổng chữ số thay tích) và kiểm tra với `(15, 3) -> 15`, `(16, 3) -> 18`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `24`, `0`, `0` (vì `45 = 9 × 5`), `99` (tích `9 × 9 = 81 = 9 × 9`, chia hết cho `9`), `20` (tích `0`). Hãy tự kiểm tra bằng code sau khi đoán. Bài 3: nhớ `p` bắt đầu từ `1`.
- **Mức 2 (vừa):** Bài 2: `23 -> 6` (không), `24 -> 8` (chia hết cho 4). Bài 4: dùng `i = n` rồi `i += 1` trong `while True`.
- **Mức 3 (sâu):** Bài 5: `count_good` là một vòng `for x in range(a, b + 1)` với `digit_product(x) % t == 0`. Với tổng chữ số, đổi `p = 1; p *= d` thành `s = 0; s += d`; lưu ý tổng `0` chỉ xảy ra với `x = 0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert digit_product(234) == 24
assert digit_product(105) == 0
assert 45 % 9 == 0
assert smallest_number_digits(99, 9) == 99     # 9*9 = 81 = 9*9 -> chia hết cho 9
assert smallest_number_digits(20, 9) == 20     # tích 0

# Bài 2: n = 23, t = 4
assert digit_product(23) == 6 and 6 % 4 != 0
assert digit_product(24) == 8 and 8 % 4 == 0
assert smallest_number_digits(23, 4) == 24

# Bài 3
def smallest_number_todo_solution(n, t):
    for i in count(n):
        p = 1
        x = i
        while x:
            p *= x % 10
            x //= 10
        if p % t == 0:
            return i

assert smallest_number_todo_solution(10, 2) == 10
assert smallest_number_todo_solution(15, 3) == 16

# Bài 4
def smallest_number_v2(n, t):
    i = n
    while True:
        p = 1
        x = i
        while x:
            p *= x % 10
            x //= 10
        if p % t == 0:
            return i
        i += 1

for n in range(1, 101):
    for t in range(1, 11):
        assert smallest_number_v2(n, t) == smallest_number_digits(n, t)

# Bài 5
def count_good(a, b, t):
    return sum(1 for x in range(a, b + 1) if digit_product(x) % t == 0)

assert count_good(10, 19, 2) == 5          # 10 (tích 0), 12, 14, 16, 18 (tích chẵn)
assert largest_at_most(15, 3) == 13

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (liệt kê từ `n` lên, kiểm tra tích chữ số)

```python
from itertools import count

class Solution:
    def smallestNumber(self, n: int, t: int) -> int:
        for i in count(n):
            p = 1
            x = i
            while x:
                p *= x % 10
                x //= 10
            if p % t == 0:
                return i
```

**Checklist**

- [ ] Tích khởi tạo `p = 1` (không phải `0`)?
- [ ] Duyệt từ **chính `n`** (đáp án có thể bằng `n`)?
- [ ] Dùng `// 10` (chia nguyên), không phải `/`?
- [ ] Chữ số `0` được nhân vào (tích thành `0`, và `0 % t == 0`)?
- [ ] Biết vì sao vòng lặp dừng (10 số liên tiếp có một số tận cùng `0`)?
- [ ] Đã thử `(10, 2) -> 10`, `(15, 3) -> 16`, `(100, 7) -> 100`, `(11, 10) -> 20`?
- [ ] Skeleton đúng `class Solution` / `def smallestNumber(self, n: int, t: int) -> int`?